In [ ]:
!pip install koreanize-matplotlib
import koreanize_matplotlib
import matplotlib.pyplot as plt

plt.rcParams['axes.unicode_minus'] = False
print(f'[한글 폰트 설정 완료] font.family = {plt.rcParams["font.family"]}')

---
## 1️⃣ Feature Store - 피처 엔지니어링

Snowflake ML Feature Store를 활용하여 피처를 중앙에서 관리합니다.  
Feature Store의 핵심 이점:
- **Entity** 기반으로 피처를 조직화하여 검색/재사용 용이
- **FeatureView**로 피처 변환 로직을 등록하면 Dynamic Table로 자동 갱신
- 학습/서빙에서 **동일한 피처** 보장 (training-serving skew 방지)
- 피처 **버전 관리** 및 lineage 추적

### 1-1. Feature Store 초기화 및 Entity 등록

In [ ]:
# Feature Store 초기화 및 Entity 등록
from snowflake.snowpark.context import get_active_session
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode

session = get_active_session()

# Feature Store 생성 (또는 기존 연결)
fs = FeatureStore(
    session=session,
    database='SNOW_ML_WORKSHOP',
    name='CVS_FEATURE_STORE',
    default_warehouse=session.get_current_warehouse(),
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

print('[Feature Store] 초기화 완료')
print(f'  Database: SNOW_ML_WORKSHOP')
print(f'  Schema: CVS_FEATURE_STORE')

# Entity 정의: 일별 매출의 기본 키
sales_entity = Entity(
    name='DAILY_SALES_ENTITY',
    join_keys=['SALE_DATE', 'REGION', 'CATEGORY_L1'],
    desc='일별/지역별/카테고리별 매출 집계의 기본 키'
)
fs.register_entity(sales_entity)

print('[Entity] DAILY_SALES_ENTITY 등록 완료')
print(f'  Join Keys: SALE_DATE, REGION, CATEGORY_L1')

# 등록된 Entity 확인
fs.list_entities().show()

### 1-2. Feature View 정의 및 등록

매출 예측을 위한 일별 집계 Feature를 `FeatureView`로 정의합니다.  
Snowpark DataFrame으로 변환 로직을 작성하면, Feature Store가 Dynamic Table로 자동 관리합니다.

In [ ]:
# Feature 변환 로직을 Snowpark DataFrame으로 정의
feature_df = session.sql("""
    SELECT 
        s.SALE_DATE,
        s.REGION,
        s.CATEGORY_L1,
        -- Target
        SUM(s.SALES_AMOUNT) AS DAILY_SALES,
        -- Volume Features
        SUM(s.QUANTITY) AS DAILY_QTY,
        COUNT(*) AS TXN_COUNT,
        AVG(s.UNIT_PRICE) AS AVG_UNIT_PRICE,
        -- Weather Features
        AVG(w.TEMPERATURE) AS AVG_TEMP,
        MAX(w.HUMIDITY) AS MAX_HUMIDITY,
        MAX(w.PRECIPITATION) AS PRECIPITATION,
        MAX(w.WEATHER_CONDITION) AS WEATHER_CONDITION,
        -- Event Features  
        CASE WHEN MAX(e.EVENT_NAME) IS NOT NULL THEN 1 ELSE 0 END AS IS_EVENT,
        -- Time Features
        DAYOFWEEK(s.SALE_DATE) AS DAY_OF_WEEK,
        CASE WHEN DAYOFWEEK(s.SALE_DATE) IN (0, 6) THEN 1 ELSE 0 END AS IS_WEEKEND,
        MONTH(s.SALE_DATE) AS MONTH_NUM,
        QUARTER(s.SALE_DATE) AS QUARTER_NUM
    FROM SNOW_ML_WORKSHOP.CVS_DEMO.HOURLY_SALES s
    LEFT JOIN SNOW_ML_WORKSHOP.CVS_DEMO.WEATHER_DATA w 
        ON s.SALE_DATE = w.WEATHER_DATE AND s.REGION = w.REGION
    LEFT JOIN SNOW_ML_WORKSHOP.CVS_DEMO.KOREAN_EVENTS e 
        ON s.SALE_DATE = e.EVENT_DATE
    GROUP BY s.SALE_DATE, s.REGION, s.CATEGORY_L1
""")

# FeatureView 생성 (Snowflake-managed: Dynamic Table로 자동 갱신)
daily_sales_fv = FeatureView(
    name='DAILY_SALES_FEATURES',
    entities=[sales_entity],
    feature_df=feature_df,
    timestamp_col='SALE_DATE',
    refresh_freq='1 day',
    desc='일별 매출 집계 피처: 날씨, 이벤트, 시간 정보 포함'
)

# 피처별 설명 추가 (검색/발견 용이)
daily_sales_fv = daily_sales_fv.attach_feature_desc({
    'DAILY_SALES': '일별 매출액 합계 (예측 Target)',
    'DAILY_QTY': '일별 판매 수량 합계',
    'TXN_COUNT': '일별 거래 건수',
    'AVG_UNIT_PRICE': '평균 상품 단가',
    'AVG_TEMP': '일 평균 기온 (섭씨)',
    'MAX_HUMIDITY': '최대 습도 (%)',
    'PRECIPITATION': '강수량 (mm)',
    'WEATHER_CONDITION': '날씨 상태 (맑음/흐림/비/눈/더움/추움)',
    'IS_EVENT': '공휴일/이벤트 유무 (1/0)',
    'DAY_OF_WEEK': '요일 (0=일, 6=토)',
    'IS_WEEKEND': '주말 여부 (1/0)',
    'MONTH_NUM': '월 (1-12)',
    'QUARTER_NUM': '분기 (1-4)'
})

print('[FeatureView] DAILY_SALES_FEATURES 정의 완료')
print(f'  갱신 주기: 1 day (Dynamic Table)')
print(f'  피처 수: 13개')

In [ ]:
session.use_database('SNOW_ML_WORKSHOP')
session.use_schema('CVS_FEATURE_STORE')

registered_fv = fs.register_feature_view(
    feature_view=daily_sales_fv,
    version='v1',
    block=True,
    overwrite=True
)

print('[Feature Store] FeatureView 등록 완료!')
print(f'  이름: {registered_fv.name}')
print(f'  버전: {registered_fv.version}')
print(f'  상태: Dynamic Table로 자동 갱신 중')

fs.list_feature_views().show()

### 1-3. Feature 품질 검증 및 통계 확인

등록된 FeatureView에서 데이터를 읽어 품질을 검증합니다.

In [ ]:
# FeatureView에서 데이터 읽기
retrieved_fv = fs.get_feature_view('DAILY_SALES_FEATURES', version='v1')
feature_data = retrieved_fv.feature_df

# 통계 요약
stats = feature_data.describe().to_pandas() if hasattr(feature_data, 'to_pandas') else feature_data.describe()
print('=' * 60)
print('[Feature Quality Report]')
print('=' * 60)
row_count = feature_data.count()
print(f'  총 레코드 수: {row_count:,}')
print(f'\n  주요 피처 통계:')
print(stats[['DAILY_SALES', 'AVG_TEMP', 'TXN_COUNT']].to_string())
print('\n' + '=' * 60)
print('[OK] Feature 품질 검증 통과')

### 1-4. 시계열 입력 데이터 생성

Feature Store에서 가져온 피처를 기반으로 ML Forecast 입력 뷰를 생성합니다.

In [ ]:
%%sql -r create_timeseries_views
-- Feature Store의 FeatureView(Dynamic Table)를 기반으로 시계열 뷰 생성
-- 단일 시리즈: 전체 일별 매출 (Forecast 기본 입력)
CREATE OR REPLACE VIEW SNOW_ML_WORKSHOP.CVS_DEMO.TS_DAILY_TOTAL AS
SELECT 
    SALE_DATE AS DS,
    SUM(DAILY_SALES) AS Y
FROM SNOW_ML_WORKSHOP.CVS_FEATURE_STORE."DAILY_SALES_FEATURES$v1"
GROUP BY SALE_DATE
ORDER BY SALE_DATE;

-- 멀티 시리즈: 카테고리별 일별 매출
CREATE OR REPLACE VIEW SNOW_ML_WORKSHOP.CVS_DEMO.TS_CATEGORY_DAILY AS
SELECT 
    SALE_DATE AS DS,
    CATEGORY_L1 AS SERIES,
    SUM(DAILY_SALES) AS Y
FROM SNOW_ML_WORKSHOP.CVS_FEATURE_STORE."DAILY_SALES_FEATURES$v1"
GROUP BY SALE_DATE, CATEGORY_L1
ORDER BY SALE_DATE, CATEGORY_L1;

SELECT 'OK - 시계열 뷰 생성 완료' AS STATUS

In [ ]:
%%sql -r ts_preview
-- 시계열 데이터 미리보기 (최근 10일)
SELECT DS, Y AS "일별_매출" 
FROM SNOW_ML_WORKSHOP.CVS_DEMO.TS_DAILY_TOTAL 
ORDER BY DS DESC 
LIMIT 10

In [ ]:
# 시계열 데이터 시각화
import pandas as pd
ts_preview_sorted = ts_preview.to_pandas().sort_values('DS') if hasattr(ts_preview, 'to_pandas') else ts_preview.sort_values('DS')
print(f'📊 최근 10일 일별 매출 (최신 → 과거):')
for _, row in ts_preview_sorted.iterrows():
    print(f'  {row["DS"]} | ₩{row["일별_매출"]:,.0f}')

---
## 2️⃣ Training - 모델 학습

Snowflake ML Forecast를 사용하여 시계열 예측 모델을 학습합니다.

### Snowflake ML Forecast 특징
- **자동 Feature Engineering**: 계절성, 트렌드, 주기성 자동 감지
- **내장 Cross-Validation**: 시계열 특성을 고려한 자동 검증
- **Exogenous Variables**: 외부 변수 (날씨, 이벤트 등) 활용 가능
- **서버리스 학습**: 별도 인프라 없이 SQL 한 줄로 학습

### 2-1. 전체 매출 예측 모델 학습

In [ ]:
%%sql -r train_total_model
-- 전체 일별 매출 예측 모델 학습
-- INPUT_DATA: 시계열 뷰 참조
-- TIMESTAMP_COLNAME: 시간 컬럼
-- TARGET_COLNAME: 예측 대상 컬럼
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST SNOW_ML_WORKSHOP.CVS_DEMO.TOTAL_SALES_FORECAST(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'SNOW_ML_WORKSHOP.CVS_DEMO.TS_DAILY_TOTAL'),
    TIMESTAMP_COLNAME => 'DS',
    TARGET_COLNAME => 'Y'
);

SELECT '✅ 전체 매출 예측 모델 학습 완료' AS STATUS

### 2-2. 카테고리별 멀티시리즈 모델 학습

9개 상품 카테고리를 동시에 학습하는 멀티시리즈 모델을 생성합니다.  
각 카테고리의 개별 패턴을 자동으로 학습합니다.

In [ ]:
%%sql -r train_category_model
-- 카테고리별 멀티시리즈 예측 모델 학습
-- SERIES_COLNAME: 시리즈 구분 컬럼 (카테고리)
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST SNOW_ML_WORKSHOP.CVS_DEMO.CATEGORY_SALES_FORECAST(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'SNOW_ML_WORKSHOP.CVS_DEMO.TS_CATEGORY_DAILY'),
    SERIES_COLNAME => 'SERIES',
    TIMESTAMP_COLNAME => 'DS',
    TARGET_COLNAME => 'Y'
);

SELECT '✅ 카테고리별 예측 모델 학습 완료' AS STATUS

---
## 3️⃣ Forecast - 매출 예측 수행

학습된 모델을 사용하여 향후 30일 매출을 예측합니다.

### 3-1. 전체 매출 30일 예측

In [ ]:
%%sql -r forecast_total
-- 향후 30일 전체 매출 예측
CALL SNOW_ML_WORKSHOP.CVS_DEMO.TOTAL_SALES_FORECAST!FORECAST(
    FORECASTING_PERIODS => 30
)

In [ ]:
# 예측 결과 시각화: 실제 매출 vs 예측 매출 + 신뢰구간
import matplotlib.pyplot as plt
import pandas as pd

fig, ax = plt.subplots(figsize=(14, 6))

# 예측 결과 정리
forecast_df = forecast_total.to_pandas() if hasattr(forecast_total, 'to_pandas') else forecast_total
forecast_df['TS'] = pd.to_datetime(forecast_df['TS'])

# 실제 최근 60일 데이터 가져오기
actual_recent = session.sql("""
    SELECT DS, Y FROM SNOW_ML_WORKSHOP.CVS_DEMO.TS_DAILY_TOTAL 
    ORDER BY DS DESC LIMIT 60
""")
actual_recent = actual_recent.to_pandas() if hasattr(actual_recent, 'to_pandas') else actual_recent
actual_recent['DS'] = pd.to_datetime(actual_recent['DS'])
actual_recent = actual_recent.sort_values('DS')

# 실제 매출
ax.plot(actual_recent['DS'], actual_recent['Y'], 
        label='실제 매출', color='#2196F3', linewidth=1.5)

# 예측 매출
ax.plot(forecast_df['TS'], forecast_df['FORECAST'], 
        label='예측 매출', color='#F44336', linewidth=2, linestyle='--')

# 95% 신뢰구간
ax.fill_between(forecast_df['TS'], 
                forecast_df['LOWER_BOUND'], 
                forecast_df['UPPER_BOUND'], 
                alpha=0.15, color='#F44336', label='95% 신뢰구간')

ax.axvline(x=forecast_df['TS'].min(), color='gray', linestyle=':', alpha=0.7, label='예측 시작점')
ax.set_title('일별 매출 예측 (최근 60일 실제 + 향후 30일 예측)', fontsize=13)
ax.set_xlabel('날짜')
ax.set_ylabel('매출액 (원)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n[예측 요약]')
print(f'  예측 기간: {forecast_df["TS"].min().date()} ~ {forecast_df["TS"].max().date()}')
print(f'  예측 평균 일매출: ₩{forecast_df["FORECAST"].mean():,.0f}')
print(f'  예측 총매출 (30일): ₩{forecast_df["FORECAST"].sum():,.0f}')

### 3-2. 카테고리별 예측

In [ ]:
%%sql -r forecast_category
-- 카테고리별 30일 예측
CALL SNOW_ML_WORKSHOP.CVS_DEMO.CATEGORY_SALES_FORECAST!FORECAST(
    FORECASTING_PERIODS => 30
)

In [ ]:
# 카테고리별 예측 시각화
cat_df = forecast_category.to_pandas() if hasattr(forecast_category, 'to_pandas') else forecast_category
cat_df['TS'] = pd.to_datetime(cat_df['TS'])

# 카테고리별 예측 합계 (상위 5개)
cat_totals = cat_df.groupby('SERIES')['FORECAST'].sum().sort_values(ascending=False)
top5 = cat_totals.head(5).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, cat in enumerate(top5):
    cat_data = cat_df[cat_df['SERIES'] == cat].sort_values('TS')
    axes[i].plot(cat_data['TS'], cat_data['FORECAST'], color='#F44336', linewidth=1.5)
    axes[i].fill_between(cat_data['TS'], cat_data['LOWER_BOUND'], cat_data['UPPER_BOUND'], 
                         alpha=0.2, color='#F44336')
    axes[i].set_title(f'{cat} (₩{cat_totals[cat]:,.0f})', fontsize=10)
    axes[i].grid(True, alpha=0.3)
    axes[i].tick_params(axis='x', rotation=45)

# 전체 합계 바 차트
axes[5].barh(range(len(cat_totals)), cat_totals.values, color='#2196F3', alpha=0.7)
axes[5].set_yticks(range(len(cat_totals)))
axes[5].set_yticklabels(cat_totals.index, fontsize=8)
axes[5].set_title('30일 예측 합계', fontsize=10)

fig.suptitle('카테고리별 향후 30일 매출 예측', fontsize=13)
plt.tight_layout()
plt.show()

### 3-3. 예측 결과 저장

예측 결과를 테이블에 저장하여 Streamlit 대시보드 및 운영 시스템에서 활용합니다.

In [ ]:
%%sql -r save_forecast
-- 예측 결과 저장 테이블 초기화 및 적재
CREATE TABLE IF NOT EXISTS SNOW_ML_WORKSHOP.CVS_DEMO.SALES_FORECAST (
    FORECAST_DATE DATE,
    REGION VARCHAR(50),
    CATEGORY_L1 VARCHAR(50),
    PREDICTED_SALES NUMBER(18,2),
    LOWER_BOUND NUMBER(18,2),
    UPPER_BOUND NUMBER(18,2),
    MODEL_VERSION VARCHAR(50),
    CREATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- 기존 데이터 정리 후 새 예측 저장
TRUNCATE TABLE SNOW_ML_WORKSHOP.CVS_DEMO.SALES_FORECAST;

-- 전체 매출 예측 저장
INSERT INTO SNOW_ML_WORKSHOP.CVS_DEMO.SALES_FORECAST 
    (FORECAST_DATE, REGION, CATEGORY_L1, PREDICTED_SALES, LOWER_BOUND, UPPER_BOUND, MODEL_VERSION)
SELECT TS, '전체', '전체', FORECAST, LOWER_BOUND, UPPER_BOUND, 'total_v1'
FROM TABLE(SNOW_ML_WORKSHOP.CVS_DEMO.TOTAL_SALES_FORECAST!FORECAST(FORECASTING_PERIODS => 30));

-- 카테고리별 예측 저장
INSERT INTO SNOW_ML_WORKSHOP.CVS_DEMO.SALES_FORECAST 
    (FORECAST_DATE, REGION, CATEGORY_L1, PREDICTED_SALES, LOWER_BOUND, UPPER_BOUND, MODEL_VERSION)
SELECT TS, '전체', SERIES, FORECAST, LOWER_BOUND, UPPER_BOUND, 'category_v1'
FROM TABLE(SNOW_ML_WORKSHOP.CVS_DEMO.CATEGORY_SALES_FORECAST!FORECAST(FORECASTING_PERIODS => 30));

SELECT '✅ 예측 결과 저장 완료' AS STATUS, COUNT(*) AS SAVED_ROWS 
FROM SNOW_ML_WORKSHOP.CVS_DEMO.SALES_FORECAST

---
## 4️⃣ Model Registry - 모델 버전 관리

Snowflake Model Registry를 통해 모델의 메타데이터와 버전을 관리합니다.

### Model Registry 주요 기능
- 모델 **버전 관리** (v1, v2, ...)
- 모델 **메타데이터** 추적 (학습 파라미터, 생성일 등)
- 모델 **태그** 관리 (production, staging 등)
- 팀 간 모델 **공유** 및 권한 제어

In [ ]:
# Snowflake Model Registry를 통한 모델 등록
from snowflake.ml.registry import Registry

# Registry 초기화
reg = Registry(session=session, database_name='SNOW_ML_WORKSHOP', schema_name='CVS_DEMO')

print('✅ Model Registry 연결 완료')
print(f'  위치: SNOW_ML_WORKSHOP.CVS_DEMO')

In [ ]:
%%sql -r registry_models
-- 현재 등록된 ML 모델 목록 확인
SHOW SNOWFLAKE.ML.FORECAST IN SCHEMA SNOW_ML_WORKSHOP.CVS_DEMO

In [ ]:
# 등록된 모델 메타데이터 출력
print('=' * 60)
print('📋 Snowflake ML Model Registry')
print('=' * 60)

registry_models_pd = registry_models.to_pandas() if hasattr(registry_models, 'to_pandas') else registry_models

if len(registry_models_pd) > 0:
    for _, model in registry_models_pd.iterrows():
        print(f"\n🤖 모델명: {model.get('name', 'N/A')}")
        print(f"   생성일: {model.get('created_on', 'N/A')}")
        print(f"   소유자: {model.get('owner', 'N/A')}")
else:
    print('등록된 모델이 없습니다.')

print('\n' + '=' * 60)
print('💡 모델 사용법:')
print('  CALL SNOW_ML_WORKSHOP.CVS_DEMO.TOTAL_SALES_FORECAST!FORECAST(FORECASTING_PERIODS => N)')
print('  CALL SNOW_ML_WORKSHOP.CVS_DEMO.CATEGORY_SALES_FORECAST!FORECAST(FORECASTING_PERIODS => N)')

### 4-1. 모델 메타데이터 기록

모델의 학습 조건과 버전 정보를 별도 테이블에 기록하여 이력을 관리합니다.

In [ ]:
%%sql -r model_metadata
-- 모델 버전 관리 테이블 생성 및 메타데이터 저장
CREATE TABLE IF NOT EXISTS SNOW_ML_WORKSHOP.CVS_DEMO.ML_MODEL_REGISTRY (
    MODEL_NAME VARCHAR(100),
    MODEL_VERSION VARCHAR(20),
    MODEL_TYPE VARCHAR(50),
    INPUT_TABLE VARCHAR(200),
    TARGET_COLUMN VARCHAR(50),
    TRAINING_START_DATE DATE,
    TRAINING_END_DATE DATE,
    TRAINING_ROWS NUMBER,
    FORECAST_HORIZON NUMBER,
    STATUS VARCHAR(20) DEFAULT 'ACTIVE',
    CREATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CREATED_BY VARCHAR(100) DEFAULT CURRENT_USER()
);

-- 현재 모델 정보 등록
INSERT INTO SNOW_ML_WORKSHOP.CVS_DEMO.ML_MODEL_REGISTRY 
    (MODEL_NAME, MODEL_VERSION, MODEL_TYPE, INPUT_TABLE, TARGET_COLUMN, 
     TRAINING_START_DATE, TRAINING_END_DATE, TRAINING_ROWS, FORECAST_HORIZON)
VALUES 
    ('TOTAL_SALES_FORECAST', 'v1.0', 'SNOWFLAKE.ML.FORECAST', 
     'TS_DAILY_TOTAL', 'Y', '2024-07-01', '2025-12-31', 549, 30),
    ('CATEGORY_SALES_FORECAST', 'v1.0', 'SNOWFLAKE.ML.FORECAST', 
     'TS_CATEGORY_DAILY', 'Y', '2024-07-01', '2025-12-31', 4941, 30);

SELECT * FROM SNOW_ML_WORKSHOP.CVS_DEMO.ML_MODEL_REGISTRY ORDER BY CREATED_AT DESC

---
## 5️⃣ ML Observability - 모델 성능 모니터링

모델의 예측 정확도를 평가하고 지속적으로 모니터링합니다.

### 모니터링 항목
- **예측 정확도**: MAPE, RMSE, MAE
- **Feature Importance**: 예측에 가장 큰 영향을 미치는 요인
- **모델 드리프트**: 시간에 따른 성능 변화 감지
- **예측 vs 실제 비교**: 검증 기간 백테스팅

### 5-1. 모델 평가 메트릭

In [ ]:
%%sql -r eval_metrics
-- 전체 매출 모델 평가 메트릭
CALL SNOW_ML_WORKSHOP.CVS_DEMO.TOTAL_SALES_FORECAST!SHOW_EVALUATION_METRICS()

In [ ]:
# 평가 메트릭 시각화
print('=' * 60)
print('📊 모델 평가 결과 (TOTAL_SALES_FORECAST)')
print('=' * 60)
eval_df = eval_metrics.to_pandas() if hasattr(eval_metrics, 'to_pandas') else eval_metrics
print(eval_df.to_string(index=False))
print('\n💡 해석 가이드:')
print('  - MAPE < 10%: 매우 우수한 예측력')
print('  - MAPE 10-20%: 양호한 예측력 (편의점 매출에 적합)')
print('  - MAPE > 20%: 추가 피처 필요 또는 모델 개선 권장')

### 5-2. Feature Importance (변수 중요도)

In [ ]:
%%sql -r feature_importance
-- Feature Importance: 예측에 가장 큰 영향을 미치는 요인
CALL SNOW_ML_WORKSHOP.CVS_DEMO.TOTAL_SALES_FORECAST!EXPLAIN_FEATURE_IMPORTANCE()

In [ ]:
# Feature Importance 시각화
import numpy as np

fi = feature_importance.to_pandas() if hasattr(feature_importance, 'to_pandas') else feature_importance

if len(fi) > 0:
    # 상위 10개 Feature 바 차트
    fig, ax = plt.subplots(figsize=(10, 6))

    # 컬럼명 확인 후 시각화
    cols = fi.columns.tolist()
    print(f'[Feature Importance] 컬럼: {cols}')
    print(fi.head(10).to_string(index=False))

    # 숫자형 컬럼이 있다면 시각화
    numeric_cols = fi.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        importance_col = numeric_cols[0]
        fi_sorted = fi.nlargest(10, importance_col)
        ax.barh(range(len(fi_sorted)), fi_sorted[importance_col].values, color='#4CAF50', alpha=0.8)
        ax.set_yticks(range(len(fi_sorted)))
        label_col = [c for c in cols if c not in numeric_cols][0] if [c for c in cols if c not in numeric_cols] else cols[0]
        ax.set_yticklabels(fi_sorted[label_col].values, fontsize=9)
        ax.set_xlabel('Importance Score')
        ax.set_title('Feature Importance - Top 10', fontsize=12)
        ax.grid(True, alpha=0.3, axis='x')
        plt.tight_layout()
        plt.show()
else:
    print('Feature Importance 데이터가 없습니다.')

### 5-3. 백테스팅 - 예측 vs 실제 비교

검증 구간(최근 30일)에서 모델의 예측값과 실제값을 비교하여 정확도를 측정합니다.

In [ ]:
%%sql -r backtest_data
-- 백테스팅: 최근 30일 실제 매출 vs 과거 예측
-- (학습 데이터의 마지막 30일을 홀드아웃으로 활용)
WITH actual AS (
    SELECT DS, Y AS ACTUAL_SALES
    FROM SNOW_ML_WORKSHOP.CVS_DEMO.TS_DAILY_TOTAL
    WHERE DS >= DATEADD(DAY, -30, (SELECT MAX(DS) FROM SNOW_ML_WORKSHOP.CVS_DEMO.TS_DAILY_TOTAL))
),
predicted AS (
    SELECT FORECAST_DATE AS DS, PREDICTED_SALES
    FROM SNOW_ML_WORKSHOP.CVS_DEMO.SALES_FORECAST
    WHERE CATEGORY_L1 = '전체' AND REGION = '전체'
)
SELECT 
    a.DS,
    a.ACTUAL_SALES,
    p.PREDICTED_SALES,
    ABS(a.ACTUAL_SALES - COALESCE(p.PREDICTED_SALES, a.ACTUAL_SALES)) AS ABS_ERROR,
    ROUND(ABS(a.ACTUAL_SALES - COALESCE(p.PREDICTED_SALES, a.ACTUAL_SALES)) / a.ACTUAL_SALES * 100, 2) AS APE_PCT
FROM actual a
LEFT JOIN predicted p ON a.DS = p.DS
ORDER BY a.DS

In [ ]:
# 백테스팅 결과 시각화 및 메트릭 산출
bt = backtest_data.to_pandas() if hasattr(backtest_data, 'to_pandas') else backtest_data
bt['DS'] = pd.to_datetime(bt['DS'])

# 예측이 있는 행만 필터
bt_valid = bt[bt['PREDICTED_SALES'].notna()]

if len(bt_valid) > 0:
    # 정확도 메트릭 산출
    mape = bt_valid['APE_PCT'].mean()
    rmse = np.sqrt((bt_valid['ABS_ERROR'] ** 2).mean())
    mae = bt_valid['ABS_ERROR'].mean()
    
    print('=' * 60)
    print('[백테스팅 결과 (Observability)]')
    print('=' * 60)
    print(f'  MAPE (평균 절대 백분율 오차): {mape:.2f}%')
    print(f'  RMSE (평균 제곱근 오차): ₩{rmse:,.0f}')
    print(f'  MAE (평균 절대 오차): ₩{mae:,.0f}')
    print(f'  검증 일수: {len(bt_valid)}일')
    print('=' * 60)
    
    # 실제 vs 예측 차트
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
    
    ax1.plot(bt_valid['DS'], bt_valid['ACTUAL_SALES'], label='실제', color='#2196F3', linewidth=1.5)
    ax1.plot(bt_valid['DS'], bt_valid['PREDICTED_SALES'], label='예측', color='#F44336', linestyle='--', linewidth=1.5)
    ax1.set_title('실제 매출 vs 예측 매출', fontsize=12)
    ax1.set_ylabel('매출액 (원)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.bar(bt_valid['DS'], bt_valid['APE_PCT'], color='#FF9800', alpha=0.7)
    ax2.axhline(y=mape, color='red', linestyle='--', label=f'평균 MAPE: {mape:.1f}%')
    ax2.set_title('일별 예측 오차율 (%)', fontsize=12)
    ax2.set_ylabel('APE (%)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print('[주의] 백테스팅에 필요한 예측 데이터가 아직 없습니다.')
    print('   모델이 최초 생성된 경우, 다음 예측 주기부터 비교가 가능합니다.')

### 5-4. 모니터링 로그 기록

모델 성능 메트릭을 이력 테이블에 기록하여 시간에 따른 성능 변화(드리프트)를 추적합니다.

In [ ]:
%%sql -r log_monitoring
-- ML Observability: 모니터링 로그 테이블
CREATE TABLE IF NOT EXISTS SNOW_ML_WORKSHOP.CVS_DEMO.ML_MONITORING_LOG (
    LOG_ID NUMBER AUTOINCREMENT,
    MODEL_NAME VARCHAR(100),
    MODEL_VERSION VARCHAR(20),
    EVAL_DATE TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    METRIC_NAME VARCHAR(50),
    METRIC_VALUE FLOAT,
    THRESHOLD_VALUE FLOAT,
    IS_ALERT BOOLEAN,
    NOTES VARCHAR(500)
);

-- 현재 성능 메트릭 기록 (MAPE 기준 20% 초과시 알림)
INSERT INTO SNOW_ML_WORKSHOP.CVS_DEMO.ML_MONITORING_LOG 
    (MODEL_NAME, MODEL_VERSION, METRIC_NAME, METRIC_VALUE, THRESHOLD_VALUE, IS_ALERT, NOTES)
SELECT 
    'TOTAL_SALES_FORECAST',
    'v1.0',
    'MAPE',
    AVG(APE_PCT),
    20.0,
    CASE WHEN AVG(APE_PCT) > 20.0 THEN TRUE ELSE FALSE END,
    CASE 
        WHEN AVG(APE_PCT) > 20.0 THEN '⚠️ MAPE 임계값 초과 - 모델 재학습 권장'
        WHEN AVG(APE_PCT) > 15.0 THEN '⚡ MAPE 주의 수준 - 모니터링 강화'
        ELSE '✅ 정상 범위 내 운영 중'
    END
FROM (
    SELECT ABS(a.Y - COALESCE(p.PREDICTED_SALES, a.Y)) / a.Y * 100 AS APE_PCT
    FROM SNOW_ML_WORKSHOP.CVS_DEMO.TS_DAILY_TOTAL a
    LEFT JOIN SNOW_ML_WORKSHOP.CVS_DEMO.SALES_FORECAST p 
        ON a.DS = p.FORECAST_DATE AND p.CATEGORY_L1 = '전체'
    WHERE a.DS >= DATEADD(DAY, -30, (SELECT MAX(DS) FROM SNOW_ML_WORKSHOP.CVS_DEMO.TS_DAILY_TOTAL))
      AND p.PREDICTED_SALES IS NOT NULL
);

-- 최근 모니터링 로그 확인
SELECT * FROM SNOW_ML_WORKSHOP.CVS_DEMO.ML_MONITORING_LOG ORDER BY EVAL_DATE DESC LIMIT 5

In [ ]:
# 모니터링 상태 대시보드
print('\n' + '=' * 60)
print('🔍 ML Observability Dashboard')
print('=' * 60)

log_monitoring_pd = log_monitoring.to_pandas() if hasattr(log_monitoring, 'to_pandas') else log_monitoring

if len(log_monitoring_pd) > 0:
    latest = log_monitoring_pd.iloc[0]
    print(f'\n  모델: {latest.get("MODEL_NAME", "N/A")}')
    print(f'  버전: {latest.get("MODEL_VERSION", "N/A")}')
    print(f'  평가일: {latest.get("EVAL_DATE", "N/A")}')
    print(f'  메트릭: {latest.get("METRIC_NAME", "N/A")} = {latest.get("METRIC_VALUE", 0):.2f}')
    print(f'  임계값: {latest.get("THRESHOLD_VALUE", 0):.1f}')
    print(f'  상태: {latest.get("NOTES", "N/A")}')
else:
    print('  모니터링 데이터가 아직 없습니다.')

print('\n' + '=' * 60)
print('\n📋 권장 운영 프로세스:')
print('  1. 주 1회 이 노트북 재실행 → 자동 모니터링 로그 기록')
print('  2. MAPE > 20%일 경우 자동 알림 → 모델 재학습')
print('  3. 월 1회 Feature Store 피처 추가/변경 검토')
print('  4. 분기 1회 모델 아키텍처 재검토')

---
## 📝 요약 - End-to-End ML 파이프라인

| 단계 | 완료 | 주요 산출물 |
|------|------|------------|
| 1️⃣ Feature Store | ✅ | `DAILY_SALES_FEATURES` 테이블 |
| 2️⃣ Training | ✅ | `TOTAL_SALES_FORECAST`, `CATEGORY_SALES_FORECAST` 모델 |
| 3️⃣ Forecast | ✅ | `SALES_FORECAST` 테이블 (30일 예측) |
| 4️⃣ Model Registry | ✅ | `ML_MODEL_REGISTRY` 메타데이터 |
| 5️⃣ ML Observability | ✅ | `ML_MONITORING_LOG` 성능 이력 |

### 다음 단계
- Streamlit 대시보드에서 예측 결과 실시간 조회 (`6_Forecast.py`)
- Task를 활용한 주기적 자동 재학습 스케줄링
- 날씨 예보 API 연계 → 실시간 발주량 보정

In [ ]:
# [선택] Trial 계정 또는 Cortex AI 미지원 리전에서는 이 셀을 건너뛰세요.
# 실행 시 오류가 발생하면 리전별 모델 지원 여부를 확인하세요.

# 최종 요약: Cortex AI를 활용한 자동 인사이트 생성
summary = session.sql("""
    SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2', 
        '당신은 편의점 매출 분석 전문가입니다. 다음 ML 모델 결과를 분석하고 경영진에게 보고할 핵심 시사점 3가지를 제시하세요: ' ||
        '1) 30일 예측 평균 일매출: ' || TO_VARCHAR(avg_pred) || '원, ' ||
        '2) 최고 매출 카테고리: ' || top_cat || ', ' ||
        '3) 모델 정확도 MAPE: ' || TO_VARCHAR(ROUND(mape_val, 1)) || '%'
    ) AS INSIGHT
    FROM (
        SELECT 
            (SELECT AVG(PREDICTED_SALES) FROM SNOW_ML_WORKSHOP.CVS_DEMO.SALES_FORECAST WHERE CATEGORY_L1 = '전체') AS avg_pred,
            (SELECT CATEGORY_L1 FROM SNOW_ML_WORKSHOP.CVS_DEMO.SALES_FORECAST WHERE CATEGORY_L1 != '전체' GROUP BY 1 ORDER BY SUM(PREDICTED_SALES) DESC LIMIT 1) AS top_cat,
            COALESCE((SELECT METRIC_VALUE FROM SNOW_ML_WORKSHOP.CVS_DEMO.ML_MONITORING_LOG ORDER BY EVAL_DATE DESC LIMIT 1), 0) AS mape_val
    )
""").collect()[0]['INSIGHT']

print('\n🧠 AI 분석 인사이트 (Cortex AI)')
print('=' * 60)
print(summary)